# Data Cleaning Exercises with Pandas

A reproducible notebook covering data audits, standardization, parsing, missing values, duplicates, validation, and a final cleaning workflow.

In [1]:
import pandas as pd
import numpy as np
pd.__version__

'2.2.3'

## 1. Create a deliberately messy dataset

In [2]:
raw = pd.DataFrame({
    " Customer Name ": [" Alice ", "BOB", "BOB", None, "Dana "],
    "Email ": ["ALICE@example.com ", "bob@example.com", "bob@example.com", "no-email", None],
    "Age": ["29", "unknown", "41", None, "150"],
    "Amount": ["₹1,200", "500", "500", "N/A", "-30"],
    "Order Date": ["2026-01-02", "bad-date", "bad-date", "2026-02-03", "2026-02-05"]
})
raw

,Customer Name,Email,Age,Amount,Order Date
0,Alice,ALICE@example.com,29,"₹1,200",2026-01-02
1,BOB,bob@example.com,unknown,500,bad-date
2,BOB,bob@example.com,41,500,bad-date
3,None,no-email,None,N/A,2026-02-03
4,Dana,None,150,-30,2026-02-05


## 2. Audit missing values and duplicates

In [3]:
audit = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing_count": raw.isna().sum(),
    "missing_percent": (raw.isna().mean() * 100).round(1)
})
audit, int(raw.duplicated().sum()), raw.shape

(                  dtype  missing_count  missing_percent
  Customer Name   object              1             20.0
 Email            object              1             20.0
 Age              object              1             20.0
 Amount           object              0              0.0
 Order Date       object              0              0.0,
 0,
 (5, 5))

## 3. Standardize column names and text

In [4]:
df = raw.copy()
df.columns = (df.columns.astype("string").str.strip().str.lower()
              .str.replace(r"\s+", "_", regex=True))
for col in ["customer_name", "email"]:
    df[col] = df[col].astype("string").str.strip()
df["customer_name"] = df["customer_name"].str.title()
df["email"] = df["email"].str.lower()
df

,customer_name,email,age,amount,order_date
0,Alice,alice@example.com,29,"₹1,200",2026-01-02
1,Bob,bob@example.com,unknown,500,bad-date
2,Bob,bob@example.com,41,500,bad-date
3,<NA>,no-email,None,N/A,2026-02-03
4,Dana,<NA>,150,-30,2026-02-05


## 4. Parse numeric and datetime columns

In [5]:
amount_text = (df["amount"].astype("string").str.replace("₹", "", regex=False)
               .str.replace(",", "", regex=False).str.strip())
df["amount"] = pd.to_numeric(amount_text, errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
invalid_dates = int(df["order_date"].isna().sum())
df[["age", "amount", "order_date"]], invalid_dates

(     age  amount order_date
 0   29.0    1200 2026-01-02
 1    NaN     500        NaT
 2   41.0     500        NaT
 3    NaN    <NA> 2026-02-03
 4  150.0     -30 2026-02-05,
 2)

## 5. Handle missing values transparently

In [6]:
df["age_was_missing"] = df["age"].isna()
age_median = df["age"].median()
df["age_filled"] = df["age"].fillna(age_median)
df["email_missing"] = df["email"].isna()
df[["age", "age_was_missing", "age_filled", "email", "email_missing"]]

,age,age_was_missing,age_filled,email,email_missing
0,29.0,False,29.0,alice@example.com,False
1,NaN,True,41.0,bob@example.com,False
2,41.0,False,41.0,bob@example.com,False
3,NaN,True,41.0,no-email,False
4,150.0,False,150.0,<NA>,True


## 6. Duplicate handling and validation flags

In [7]:
before_rows = len(df)
df_no_duplicates = df.drop_duplicates()
removed_duplicates = before_rows - len(df_no_duplicates)
df_no_duplicates["invalid_age"] = ~df_no_duplicates["age"].between(0, 120) & df_no_duplicates["age"].notna()
df_no_duplicates["invalid_email"] = ~df_no_duplicates["email"].str.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", na=False)
df_no_duplicates[["customer_name", "email", "age", "invalid_age", "invalid_email"]], removed_duplicates

(  customer_name              email    age  invalid_age  invalid_email
 0         Alice  alice@example.com   29.0        False          False
 1           Bob    bob@example.com    NaN        False          False
 2           Bob    bob@example.com   41.0        False          False
 3          <NA>           no-email    NaN        False           True
 4          Dana               <NA>  150.0         True           True,
 0)

## 7. IQR outlier flagging

In [8]:
values = pd.Series([10, 12, 13, 14, 15, 16, 18, 500], name="amount")
q1, q3 = values.quantile([0.25, 0.75])
iqr = q3 - q1
outlier_mask = (values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)
pd.DataFrame({"amount": values, "is_outlier": outlier_mask})

,amount,is_outlier
0,10,False
1,12,False
2,13,False
3,14,False
4,15,False
5,16,False
6,18,False
7,500,True


## 8. Quality report

In [9]:
def quality_report(frame):
    return {
        "rows": int(len(frame)),
        "columns": int(frame.shape[1]),
        "duplicate_rows": int(frame.duplicated().sum()),
        "missing_cells": int(frame.isna().sum().sum())
    }
quality_report(raw), quality_report(df_no_duplicates)

({'rows': 5, 'columns': 5, 'duplicate_rows': 0, 'missing_cells': 3},
 {'rows': 5, 'columns': 10, 'duplicate_rows': 0, 'missing_cells': 7})

## 9. Reusable cleaning function

In [10]:
def clean_orders(frame):
    result = frame.copy()
    before_rows = len(result)
    result.columns = (result.columns.astype("string").str.strip().str.lower()
                      .str.replace(r"\s+", "_", regex=True))
    if "email" in result:
        result["email"] = result["email"].astype("string").str.strip().str.lower()
    if "amount" in result:
        text = (result["amount"].astype("string").str.replace("₹", "", regex=False)
                .str.replace(",", "", regex=False).str.strip())
        result["amount"] = pd.to_numeric(text, errors="coerce")
    if "order_date" in result:
        result["order_date"] = pd.to_datetime(result["order_date"], errors="coerce")
    result = result.drop_duplicates()
    metrics = {
        "rows_before": before_rows,
        "rows_after": len(result),
        "missing_cells_after": int(result.isna().sum().sum())
    }
    return result, metrics

cleaned, metrics = clean_orders(raw)
cleaned, metrics

(  customer_name              email      age  amount order_date
 0        Alice   alice@example.com       29    1200 2026-01-02
 1           BOB    bob@example.com  unknown     500        NaT
 2           BOB    bob@example.com       41     500        NaT
 3          None           no-email     None    <NA> 2026-02-03
 4         Dana                <NA>      150     -30 2026-02-05,
 {'rows_before': 5, 'rows_after': 5, 'missing_cells_after': 6})

## 10. Export cleaned data

In [11]:
# Example export; writes a CSV in the notebook's working directory.
# cleaned.to_csv("cleaned_orders.csv", index=False)
print("Ready to export:", cleaned.shape, "rows/columns")

Ready to export: (5, 5) rows/columns
